In [1]:
!pip install --force-reinstall numpy==1.26.4 pandas==2.2.3

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for numpy==1.26.4 from https://files.pythonhosted.org/packages/3f/6b/5610004206cf7f8e7ad91c5a85a8c71b2f2f8051a0c0c4d5916b76d6cbb2/numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Obtaining dependency information for pandas==2.2.3 from https://files.pythonhosted.org/packages/ed/8c/87ddf1fcb55d11f9f847e3c69bb1c6f8e46e2f40ab1a2d2abadb2401b007/pandas-2.2.3-cp311-cp311-win_amd64.whl.metadata
  Using cached pandas-2.2.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Obtaining dependency information for python-dateutil>=2.8.2 from https://files.pythonhosted.org/packages/ec/57/56b9bcc3c9c6a792fcbaf139543cee77261f3651ca9da0c93f5c1221264b/python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Obtaining dependency information for p

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\sushm\\AppData\\Roaming\\Python\\Python311\\site-packages\\numpy-1.26.4.dist-info\\INSTALLER4l6jut0o.tmp'



In [2]:
import pandas as pd

# Create dataset
data = pd.DataFrame([
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
], columns=[
    'Sky', 'AirTemp', 'Humidity', 'Wind',
    'Water', 'Forecast', 'EnjoySport'
])

print("Dataset:")
print(data)

attributes = list(data.columns[:-1])
target = data.columns[-1]


# --------------------------------------------------
# FIND-S ALGORITHM
# --------------------------------------------------

def find_s(df):

    hypothesis = ['0'] * len(attributes)

    for _, row in df.iterrows():

        # Consider only positive examples
        if row[target] == 'Yes':

            for j in range(len(attributes)):

                if hypothesis[j] == '0':
                    hypothesis[j] = row[attributes[j]]

                elif hypothesis[j] != row[attributes[j]]:
                    hypothesis[j] = '?'

    return hypothesis


final_hypothesis = find_s(data)

print("\nMost Specific Hypothesis (Find-S):")
print(final_hypothesis)


# --------------------------------------------------
# CANDIDATE ELIMINATION ALGORITHM
# --------------------------------------------------

def candidate_elimination(df):

    # Most specific boundary
    S = ['0'] * len(attributes)

    # Most general boundary
    G = [['?'] * len(attributes)]

    for _, row in df.iterrows():

        example = [row[attr] for attr in attributes]

        # ------------------------------------------
        # POSITIVE EXAMPLE
        # ------------------------------------------
        if row[target] == 'Yes':

            # Remove hypotheses from G that
            # do not cover the positive example
            G = [
                g for g in G
                if all(
                    g[j] == '?' or g[j] == example[j]
                    for j in range(len(attributes))
                )
            ]

            # Generalize S
            for j in range(len(attributes)):

                if S[j] == '0':
                    S[j] = example[j]

                elif S[j] != example[j]:
                    S[j] = '?'

        # ------------------------------------------
        # NEGATIVE EXAMPLE
        # ------------------------------------------
        else:

            new_G = []

            for g in G:

                # Check whether G hypothesis covers
                # the negative example
                if all(
                    g[j] == '?' or g[j] == example[j]
                    for j in range(len(attributes))
                ):

                    # Specialize G
                    for j in range(len(attributes)):

                        if g[j] == '?':

                            for value in df[attributes[j]].unique():

                                if value != example[j]:

                                    new_hypothesis = g.copy()
                                    new_hypothesis[j] = value

                                    # Check consistency with S
                                    if all(
                                        S[k] == '0'
                                        or S[k] == '?'
                                        or new_hypothesis[k] == '?'
                                        or S[k] == new_hypothesis[k]
                                        for k in range(len(attributes))
                                    ):
                                        new_G.append(new_hypothesis)

                else:
                    new_G.append(g)

            G = new_G

    return S, G


S_final, G_final = candidate_elimination(data)

print("\nCandidate Elimination Results:")

print("\nS (Most Specific Boundary):")
print(S_final)

print("\nG (Most General Boundary):")
for g in G_final:
    print(g)

C:\Users\sushm\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Dataset:
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High  Strong  Warm   Change         No
3  Sunny    Warm     High  Strong  Cool   Change        Yes

Most Specific Hypothesis (Find-S):
['Sunny', 'Warm', '?', 'Strong', '?', '?']

Candidate Elimination Results:

S (Most Specific Boundary):
['Sunny', 'Warm', '?', 'Strong', '?', '?']

G (Most General Boundary):
['Sunny', '?', '?', '?', '?', '?']
['?', 'Warm', '?', '?', '?', '?']
